In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm


# 버전에 따른 폰트 캐시 재생성
import os
import shutil

# 방법 1: 캐시 디렉토리 삭제 (모든 버전 호환)
cache_dir = matplotlib.get_cachedir()
print(f"캐시 디렉토리: {cache_dir}")

# 캐시 삭제
try:
    shutil.rmtree(cache_dir)
    print("폰트 캐시 삭제 완료")
except:
    print("캐시 삭제 실패 또는 이미 삭제됨")

# 설치된 폰트 확인
font_list = [f.name for f in fm.fontManager.ttflist]
print('NanumGothic' in font_list)  # True가 나와야 함

# 한글 폰트 설정
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

    # # 파일 경로 리스트
    # file_paths = [
    #     '../../../data/kweather/version3_250812/안전공조시스템 데이터_250804-250812_거실.csv',
    #     '../../../data/kweather/version3_250812/안전공조시스템 데이터_250804-250812_방1.csv',
    #     '../../../data/kweather/version3_250812/안전공조시스템 데이터_250804-250812_방2.csv'
    # ]
    

In [ ]:
# 변수 타입 정의 (사전 정의)
PREDEFINED_NUMERICAL = [
    '실내미세먼지(㎍/㎥)', '실내초미세먼지(㎍/㎥)', '실내이산화탄소(ppm)', 
    '실내휘발성유기화합물(ppb)', '실내온도(℃)', '실내습도(%)', 
    '실외미세먼지(㎍/㎥)', '실외초미세먼지(㎍/㎥)', '실외온도(℃)', 
    '실외습도(%)', '환기누적전력량(Wh)', '환기유효전력(W)', 
    '냉난방설정온도(℃)', '냉난방누적전력량(Wh)', '냉난방유효전력(W)'
]

PREDEFINED_CATEGORICAL = [
    '환기전원', '환기풍량', '환기운전모드', 
    '냉난방전원', '냉난방풍량', '냉난방운전모드'
]

# 범주형 변수의 정의된 클래스
CATEGORICAL_CLASSES = {
    '환기전원': {
        'OFF': 'OFF',
        'ON': 'ON'
    },
    '환기풍량': {
        '0': '0',
        '1': '1',
        '2': '2',
        '3': '3',
        '4': '4',
        '5': '5',
        '6': '6',
        '0.0': '0',
        '1.0': '1',  # 숫자형으로도 올 수 있음
        '2.0': '2',
        '3.0': '3',
        '4.0': '4',
        '5.0': '5',
        '6.0': '6'
    },
    '환기운전모드': {
        '환기': '환기청정',
        '공기': '공기청정',
        '바이': '바이패스',
        '꺼짐': '꺼짐',
        '환기청정': '환기청정',
        '공기청정': '공기청정',
        '바이패스': '바이패스',
    },
    '냉난방전원': {
        'OFF': 'OFF',
        'ON': 'ON'
    },
    '냉난방풍량': {
        '자동': '자동',
        '약풍': '약풍',
        '중풍': '중풍',
        '강풍': '강풍',
        '자동풍': '자동풍',
        '미약풍': '미약풍',
        '파워풍': '파워풍',
    },
    '냉난방운전모드': {
        '자동': '자동',
        '냉방': '냉방',
        '제습': '제습',
        '송풍': '송풍',
        '난방': '난방'
    }
}

# 시간 컬럼명 정의
TIME_COLUMN = '데이터 시간'

def preprocess_dataframe(df):
    """
    데이터프레임 전처리: "-" 를 NaN으로 변환
    """
    # "-" 를 NaN으로 변환
    df = df.replace('-', np.nan)
    
    # 수치형 컬럼 변환 시도
    for col in df.columns:
        if col != TIME_COLUMN and col not in PREDEFINED_CATEGORICAL:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except:
                pass
    
    return df

def analyze_dataframe(df, file_name="", location_name=""):
    """
    데이터프레임에 대한 포괄적인 분석 수행
    """
    
    print("="*80)
    print(f"📊 데이터 분석: {file_name}")
    if location_name:
        print(f"📍 Location: {location_name}")
    print("="*80)
    
    # 데이터 전처리
    print("\n⚙️ Preprocessing data (converting '-' to NaN)...")
    df = preprocess_dataframe(df)
    
    # =====================================
    # 1. Basic Information
    # =====================================
    print(f"\n📌 1. Basic Information - {location_name if location_name else file_name}")
    print("-"*50)
    
    print(f"• Shape: {df.shape}")
    print(f"• Rows: {df.shape[0]:,}")
    print(f"• Columns: {df.shape[1]}")
    print(f"• Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # 데이터 타입 요약
    dtype_counts = df.dtypes.value_counts()
    print(f"\n• Data Types Summary:")
    for dtype, count in dtype_counts.items():
        print(f"  - {dtype}: {count} columns")
    
    # =====================================
    # 2. Time Series Analysis
    # =====================================
    print(f"\n📌 2. Time Series Analysis")
    print("-"*50)
    
    # '데이터 시간' 컬럼 확인
    if TIME_COLUMN in df.columns:
        # datetime 변환
        try:
            df[TIME_COLUMN] = pd.to_datetime(df[TIME_COLUMN])
            print(f"✓ Time column '{TIME_COLUMN}' converted to datetime")
            
            # 역순 정렬
            df = df.sort_values(TIME_COLUMN, ascending=False).reset_index(drop=True)
            
            print(f"\n• Time Range:")
            print(f"  - Start: {df[TIME_COLUMN].iloc[-1]}")
            print(f"  - End: {df[TIME_COLUMN].iloc[0]}")
            print(f"  - Duration: {df[TIME_COLUMN].iloc[0] - df[TIME_COLUMN].iloc[-1]}")
            
            # 시간 간격 분석
            time_diff = df[TIME_COLUMN].diff().dropna()
            time_diff_abs = time_diff.abs()
            
            print(f"\n• Time Interval Analysis:")
            print(f"  - Mean interval: {time_diff.mean()}")
            print(f"  - Min interval: {time_diff_abs.min()}")
            print(f"  - Max interval: {time_diff_abs.max()}")
            print(f"  - Std deviation: {time_diff.std()}")
            
            # 일정한 간격인지 확인
            time_diff_minutes = time_diff_abs.dt.total_seconds() / 60
            unique_intervals = time_diff_minutes.value_counts()
            
            print(f"\n• Interval Consistency Check:")
            print(f"  - Number of unique intervals: {len(unique_intervals)}")
            
            if len(unique_intervals) == 1:
                interval_value = unique_intervals.index[0]
                print(f"  ✓ All data points have consistent interval ({interval_value:.1f} minutes)")
            else:
                print(f"  ✗ Irregular intervals detected")
                print(f"  - Top 3 most frequent intervals:")
                for interval, count in unique_intervals.head(3).items():
                    print(f"    • {interval:.1f} min: {count} times ({count/len(time_diff)*100:.1f}%)")
        except Exception as e:
            print(f"✗ Failed to convert '{TIME_COLUMN}' to datetime: {e}")
    else:
        print(f"⚠️ Time column '{TIME_COLUMN}' not found in dataset")
    
    # =====================================
    # 3. Duplicate Analysis
    # =====================================
    print(f"\n📌 3. Duplicate Analysis")
    print("-"*50)
    
    # 전체 행 중복 확인
    duplicated_rows = df.duplicated().sum()
    print(f"\n• Completely Duplicated Rows: {duplicated_rows} ({duplicated_rows/len(df)*100:.2f}%)")
    
    if duplicated_rows > 0:
        dup_indices = df[df.duplicated()].index.tolist()
        print(f"  - Duplicate row indices (first 10): {dup_indices[:10]}")
        if len(dup_indices) > 10:
            print(f"  - ... and {len(dup_indices)-10} more")
    
    # 시간 컬럼 중복 확인
    if TIME_COLUMN in df.columns:
        time_duplicates = df[TIME_COLUMN].duplicated().sum()
        print(f"\n• Time Column Duplicates: {time_duplicates} ({time_duplicates/len(df)*100:.2f}%)")
        
        if time_duplicates > 0:
            dup_times = df[df[TIME_COLUMN].duplicated()][TIME_COLUMN].value_counts()
            print(f"  - Top 3 most duplicated timestamps:")
            for time_val, count in dup_times.head(3).items():
                print(f"    • {time_val}: {count} duplicates")
    
    # =====================================
    # 4. Missing Values Analysis
    # =====================================
    print(f"\n📌 4. Missing Values Analysis")
    print("-"*50)
    
    missing_counts = df.isnull().sum()
    total_missing = missing_counts.sum()
    
    print(f"\n• Total Missing Values: {total_missing} ({total_missing/(len(df)*len(df.columns))*100:.2f}%)")
    print(f"  (Note: '-' values have been converted to NaN)")
    
    if total_missing > 0:
        print("\n• Missing Values by Column:")
        missing_data = pd.DataFrame({
            'Missing Count': missing_counts[missing_counts > 0],
            'Missing %': (missing_counts[missing_counts > 0] / len(df)) * 100
        }).sort_values('Missing Count', ascending=False)
        
        for col, row in missing_data.iterrows():
            print(f"  - {col}: {row['Missing Count']:.0f} ({row['Missing %']:.2f}%)")
        
        # 행별 결측치 패턴
        missing_per_row = df.isnull().sum(axis=1)
        rows_with_missing = (missing_per_row > 0).sum()
        print(f"\n• Rows with Missing Values: {rows_with_missing} ({rows_with_missing/len(df)*100:.2f}%)")
        
        # 결측치 패턴 분포
        if rows_with_missing > 0:
            pattern_counts = missing_per_row[missing_per_row > 0].value_counts()
            print(f"\n• Missing Pattern Distribution (# of missing values per row):")
            for n_missing, count in pattern_counts.head(5).items():
                print(f"  - {n_missing} missing values: {count} rows")
    else:
        print("✓ No missing values found")
    
    # =====================================
    # 5. Variable Type Classification (Predefined)
    # =====================================
    print(f"\n📌 5. Variable Type Classification (Predefined)")
    print("-"*50)
    
    # 실제 존재하는 컬럼 확인
    actual_numerical = [col for col in PREDEFINED_NUMERICAL if col in df.columns]
    actual_categorical = [col for col in PREDEFINED_CATEGORICAL if col in df.columns]
    
    # 정의되지 않은 컬럼 찾기
    all_predefined = set(PREDEFINED_NUMERICAL + PREDEFINED_CATEGORICAL + [TIME_COLUMN])
    undefined_columns = [col for col in df.columns if col not in all_predefined]
    
    print(f"\n• Variable Classification Summary:")
    print(f"  - Numerical Variables: {len(actual_numerical)}/{len(PREDEFINED_NUMERICAL)} found")
    print(f"  - Categorical Variables: {len(actual_categorical)}/{len(PREDEFINED_CATEGORICAL)} found")
    print(f"  - Undefined Variables: {len(undefined_columns)}")
    
    # =====================================
    # 6. Numerical Variables Analysis
    # =====================================
    print(f"\n📌 6. Numerical Variables Analysis")
    print("-"*50)
    
    if actual_numerical:
        print("\n• Basic Statistics Summary:")
        for col in actual_numerical:
            if col in df.columns:
                # NaN이 아닌 값만 사용하여 통계 계산
                valid_data = df[col].dropna()
                
                if len(valid_data) > 0:
                    print(f"\n  [{col}]")
                    print(f"    - Valid Count: {len(valid_data):,} ({len(valid_data)/len(df)*100:.1f}%)")
                    print(f"    - Missing: {len(df) - len(valid_data):,} ({(len(df) - len(valid_data))/len(df)*100:.1f}%)")
                    print(f"    - Mean: {valid_data.mean():.2f}")
                    print(f"    - Std: {valid_data.std():.2f}")
                    print(f"    - Min: {valid_data.min():.2f}")
                    print(f"    - Max: {valid_data.max():.2f}")
                    print(f"    - 25%: {valid_data.quantile(0.25):.2f}")
                    print(f"    - 50% (Median): {valid_data.quantile(0.50):.2f}")
                    print(f"    - 75%: {valid_data.quantile(0.75):.2f}")
                    
                    # 이상치 검출 (IQR)
                    Q1 = valid_data.quantile(0.25)
                    Q3 = valid_data.quantile(0.75)
                    IQR = Q3 - Q1
                    lower_bound = Q1 - 1.5 * IQR
                    upper_bound = Q3 + 1.5 * IQR
                    outliers = valid_data[(valid_data < lower_bound) | (valid_data > upper_bound)]
                    
                    if len(outliers) > 0:
                        print(f"    - Outliers: {len(outliers)} ({len(outliers)/len(valid_data)*100:.2f}% of valid data)")
                        print(f"    - Normal range: [{lower_bound:.2f}, {upper_bound:.2f}]")
                else:
                    print(f"\n  [{col}]")
                    print(f"    - All values are missing (100% NaN)")
    else:
        print("• No predefined numerical variables found in the dataset")
    
    # =====================================
    # 7. Categorical Variables Analysis
    # =====================================
    print(f"\n📌 7. Categorical Variables Analysis")
    print("-"*50)
    
    if actual_categorical:
        for col in actual_categorical:
            if col in df.columns:
                print(f"\n• [{col}]")
                
                # NaN 개수 먼저 확인
                missing = df[col].isnull().sum()
                valid_count = len(df) - missing
                
                if valid_count > 0:
                    unique_count = df[col].nunique()
                    print(f"  - Valid Count: {valid_count:,} ({valid_count/len(df)*100:.1f}%)")
                    print(f"  - Missing: {missing:,} ({missing/len(df)*100:.1f}%)")
                    print(f"  - Unique values in data: {unique_count}")
                    
                    # 정의된 클래스가 있는 경우
                    if col in CATEGORICAL_CLASSES:
                        defined_classes = CATEGORICAL_CLASSES[col]
                        print(f"  - Defined classes: {len(set(defined_classes.values()))} types")
                        
                        # 실제 데이터의 값 분포
                        value_counts = df[col].value_counts()
                        actual_values = set(value_counts.index.astype(str))
                        
                        # 정의된 모든 고유 클래스 (중복 제거)
                        unique_defined_classes = {}
                        for key, val in defined_classes.items():
                            if val not in unique_defined_classes.values():
                                unique_defined_classes[key] = val
                        
                        print(f"\n  - Class Distribution (Defined vs Actual):")
                        
                        # 정의된 각 클래스에 대해 확인
                        for class_key, class_name in unique_defined_classes.items():
                            # 해당 클래스에 매칭되는 모든 키 찾기
                            matching_keys = [k for k, v in defined_classes.items() if v == class_name]
                            
                            # 실제 데이터에서 해당 값 찾기
                            found = False
                            for key in matching_keys:
                                if str(key) in actual_values:
                                    count = value_counts[value_counts.index.astype(str) == str(key)].sum()
                                    percentage = count / len(df) * 100
                                    valid_percentage = count / valid_count * 100
                                    print(f"    • {class_name} ({key}): {count:,} ({percentage:.2f}% of total, {valid_percentage:.2f}% of valid)")
                                    found = True
                                    break
                            
                            if not found:
                                print(f"    • {class_name} ({class_key}): 0 (0.00% - NOT FOUND IN DATA)")
                        
                        # 정의되지 않은 값이 데이터에 있는지 확인
                        undefined_values = []
                        for value in actual_values:
                            if str(value) not in defined_classes.keys():
                                count = value_counts[value_counts.index.astype(str) == str(value)].sum()
                                undefined_values.append((value, count))
                        
                        if undefined_values:
                            print(f"\n  - Undefined values found in data:")
                            for value, count in undefined_values:
                                percentage = count / len(df) * 100
                                print(f"    ⚠️ '{value}': {count:,} ({percentage:.2f}% of total)")
                    else:
                        # 정의된 클래스가 없는 경우 일반적인 분포 표시
                        value_counts = df[col].value_counts()
                        print(f"  - Value distribution:")
                        
                        for value, count in value_counts.items():
                            percentage = count / len(df) * 100
                            valid_percentage = count / valid_count * 100
                            print(f"    • {value}: {count:,} ({percentage:.2f}% of total, {valid_percentage:.2f}% of valid)")
                    
                    # 최빈값
                    value_counts = df[col].value_counts()
                else:
                    print(f"  - All values are missing (100% NaN)")
    else:
        print("• No predefined categorical variables found in the dataset")
    
    # =====================================
    # 8. Undefined Variables
    # =====================================
    if undefined_columns:
        print(f"\n📌 8. Undefined Variables Analysis")
        print("-"*50)
        
        print(f"\n• Found {len(undefined_columns)} undefined columns:")
        for col in undefined_columns:
            dtype = df[col].dtype
            unique_count = df[col].nunique()
            missing = df[col].isnull().sum()
            
            print(f"\n  [{col}]")
            print(f"    - Data type: {dtype}")
            print(f"    - Unique values: {unique_count}")
            print(f"    - Missing: {missing} ({missing/len(df)*100:.2f}%)")
            
            # 값 분포 (10개 이하면 모두 표시)
            if unique_count <= 10 and unique_count > 0:
                value_counts = df[col].value_counts()
                print(f"    - Values:")
                for value, count in value_counts.items():
                    print(f"      • {value}: {count} ({count/len(df)*100:.2f}%)")
    
    # =====================================
    # 9. Outlier Summary
    # =====================================
    print(f"\n📌 9. Outlier Detection Summary (IQR Method)")
    print("-"*50)
    
    outlier_summary = {}
    total_outliers = 0
    
    for col in actual_numerical:
        if col in df.columns:
            valid_data = df[col].dropna()
            
            if len(valid_data) > 0:
                Q1 = valid_data.quantile(0.25)
                Q3 = valid_data.quantile(0.75)
                IQR = Q3 - Q1
                
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR
                
                outliers = valid_data[(valid_data < lower_bound) | (valid_data > upper_bound)]
                outlier_count = len(outliers)
                
                if outlier_count > 0:
                    outlier_summary[col] = {
                        'count': outlier_count,
                        'ratio': outlier_count / len(valid_data) * 100,
                        'lower_bound': lower_bound,
                        'upper_bound': upper_bound
                    }
                    total_outliers += outlier_count
    
    print(f"\n• Total Outlier Occurrences: {total_outliers} (with overlaps)")
    print(f"• Columns with Outliers: {len(outlier_summary)}/{len(actual_numerical)}")
    
    if outlier_summary:
        print("\n• Outlier Details by Column:")
        sorted_outliers = sorted(outlier_summary.items(), key=lambda x: x[1]['count'], reverse=True)
        
        for col, info in sorted_outliers:
            print(f"  - {col}: {info['count']} outliers ({info['ratio']:.2f}% of valid data)")
    
    # =====================================
    # 10. Data Quality Score
    # =====================================
    print(f"\n📌 10. Data Quality Score")
    print("-"*50)
    
    # 품질 점수 계산
    completeness = (1 - total_missing / (len(df) * len(df.columns))) * 100
    uniqueness = (1 - duplicated_rows / len(df)) * 100
    
    # 일관성 점수 (시간 간격 기준)
    consistency = 100.0
    if TIME_COLUMN in df.columns and 'unique_intervals' in locals():
        if len(unique_intervals) > 1:
            # 가장 빈번한 간격의 비율을 일관성 점수로 사용
            most_common_ratio = unique_intervals.iloc[0] / len(time_diff) * 100
            consistency = most_common_ratio
    
    overall_quality = (completeness + uniqueness + consistency) / 3
    
    print(f"• Completeness Score: {completeness:.2f}%")
    print(f"• Uniqueness Score: {uniqueness:.2f}%")
    print(f"• Consistency Score: {consistency:.2f}%")
    print(f"• Overall Quality Score: {overall_quality:.2f}%")
    
    print("\n" + "="*80)
    print("✅ Analysis Complete!")
    print("="*80)
    
    # 결과 반환
    return {
        'dataframe': df,
        'time_column': TIME_COLUMN,
        'duplicated_rows': duplicated_rows,
        'missing_total': total_missing,
        'outlier_summary': outlier_summary,
        'variable_types': {
            'numerical': actual_numerical,
            'categorical': actual_categorical,
            'undefined': undefined_columns
        },
        'quality_scores': {
            'completeness': completeness,
            'uniqueness': uniqueness,
            'consistency': consistency,
            'overall': overall_quality
        }
    }

def analyze_multiple_files(file_paths, location_names=None):
    """
    여러 파일을 각각 분석
    """
    results = {}
    
    if location_names is None:
        location_names = ['Living Room', 'Room 1', 'Room 2']
    
    for i, file_path in enumerate(file_paths):
        print(f"\n📂 Loading file: {file_path}")
        
        try:
            # CSV 파일 로드 (na_values='-' 옵션 추가)
            df = pd.read_csv(file_path, encoding='utf-8', na_values='-')
            print(f"✓ Successfully loaded: {df.shape}")
            
            # 파일명 추출
            file_name = file_path.split('/')[-1].split('\\')[-1]
            
            # 위치 이름
            location = location_names[i] if i < len(location_names) else f"Location_{i+1}"
            
            # 분석 수행
            result = analyze_dataframe(df, file_name, location)
            results[location] = result
            
        except Exception as e:
            print(f"✗ Failed to load file: {e}")
    
    return results

def compare_results(results):
    """
    여러 분석 결과를 비교
    """
    print("\n" + "="*80)
    print("📊 Comparative Analysis Across Files")
    print("="*80)
    
    # 기본 정보 비교
    print("\n📌 Basic Information Comparison")
    print("-"*50)
    
    for location, result in results.items():
        df = result['dataframe']
        print(f"\n• {location}:")
        print(f"  - Shape: {df.shape}")
        print(f"  - Quality Score: {result['quality_scores']['overall']:.2f}%")
        print(f"  - Missing Values: {result['missing_total']}")
        print(f"  - Duplicated Rows: {result['duplicated_rows']}")
    
    # 변수 존재 여부 비교
    print("\n📌 Variable Presence Comparison")
    print("-"*50)
    
    all_numerical = set()
    all_categorical = set()
    
    for location, result in results.items():
        all_numerical.update(result['variable_types']['numerical'])
        all_categorical.update(result['variable_types']['categorical'])
    
    print("\n• Numerical Variables Presence:")
    for var in sorted(all_numerical):
        presence = []
        for location, result in results.items():
            if var in result['variable_types']['numerical']:
                presence.append(location)
        print(f"  - {var}: {', '.join(presence)}")
    
    print("\n• Categorical Variables Presence:")
    for var in sorted(all_categorical):
        presence = []
        for location, result in results.items():
            if var in result['variable_types']['categorical']:
                presence.append(location)
        print(f"  - {var}: {', '.join(presence)}")

def analyze_categorical_classes(df, location_name=""):
    """
    범주형 변수의 정의된 클래스와 실제 데이터 비교 분석
    """
    print("\n" + "="*80)
    print(f"📊 Categorical Classes Analysis - {location_name}")
    print("="*80)
    
    for col in PREDEFINED_CATEGORICAL:
        if col in df.columns and col in CATEGORICAL_CLASSES:
            print(f"\n📌 [{col}]")
            print("-"*50)
            
            # 데이터 준비
            valid_data = df[col].dropna()
            missing_count = len(df) - len(valid_data)
            
            print(f"• Data Overview:")
            print(f"  - Total rows: {len(df):,}")
            print(f"  - Valid data: {len(valid_data):,} ({len(valid_data)/len(df)*100:.1f}%)")
            print(f"  - Missing: {missing_count:,} ({missing_count/len(df)*100:.1f}%)")
            
            if len(valid_data) > 0:
                # 정의된 클래스
                defined_classes = CATEGORICAL_CLASSES[col]
                unique_classes = {}
                for key, val in defined_classes.items():
                    # 중복된 값 제거 (예: 'Off'가 여러 키에 매핑)
                    if val not in unique_classes.values():
                        unique_classes[key] = val
                
                print(f"\n• Defined Classes: {len(unique_classes)} types")
                
                # 실제 데이터 값
                value_counts = valid_data.value_counts()
                actual_values = set(value_counts.index.astype(str))
                
                print(f"• Actual Unique Values: {len(actual_values)}")
                
                # 클래스별 통계
                print(f"\n• Class Statistics:")
                class_found = {}
                total_found = 0
                
                for class_key, class_name in unique_classes.items():
                    # 해당 클래스에 매칭되는 모든 키 찾기
                    matching_keys = [k for k, v in defined_classes.items() if v == class_name]
                    
                    # 실제 데이터에서 찾기 (float와 int 변환 고려)
                    found_count = 0
                    found_key = None
                    
                    for key in matching_keys:
                        # 다양한 형태로 매칭 시도
                        possible_keys = [
                            str(key),  # '1'
                            str(float(key)) if key.isdigit() else key,  # '1.0'
                            str(int(float(key))) if key.replace('.','').isdigit() else key  # '1'
                        ]
                        
                        for pk in possible_keys:
                            if pk in actual_values:
                                found_count = value_counts[value_counts.index.astype(str) == pk].sum()
                                found_key = pk
                                break
                        
                        if found_count > 0:
                            break
                    
                    if found_count > 0:
                        percentage = found_count / len(df) * 100
                        valid_percentage = found_count / len(valid_data) * 100
                        print(f"  ✓ {class_name:15s} (found as '{found_key}'): {found_count:6,} ({percentage:6.2f}% of total, {valid_percentage:6.2f}% of valid)")
                        class_found[class_name] = found_count
                        total_found += found_count
                    else:
                        print(f"  ✗ {class_name:15s} (key={class_key:5s}): {0:6,} (  0.00% - NOT FOUND)")
                        class_found[class_name] = 0
                
                # 정의되지 않은 값 확인 (float 변환 고려)
                undefined_values = []
                undefined_total = 0
                for value in actual_values:
                    # 값이 정의된 클래스에 있는지 확인 (float 변환 고려)
                    is_defined = False
                    
                    # 직접 매칭
                    if str(value) in defined_classes.keys():
                        is_defined = True
                    
                    # float를 int로 변환하여 매칭 (예: '1.0' -> '1')
                    if not is_defined and '.' in str(value):
                        try:
                            int_value = str(int(float(value)))
                            if int_value in defined_classes.keys():
                                is_defined = True
                        except:
                            pass
                    
                    if not is_defined:
                        count = value_counts[value_counts.index.astype(str) == str(value)].sum()
                        undefined_values.append((value, count))
                        undefined_total += count
                
                if undefined_values:
                    print(f"\n• ⚠️ Undefined Values Found:")
                    for value, count in sorted(undefined_values, key=lambda x: x[1], reverse=True):
                        percentage = count / len(df) * 100
                        valid_percentage = count / len(valid_data) * 100
                        print(f"  - '{value}': {count:,} ({percentage:.2f}% of total, {valid_percentage:.2f}% of valid)")
                
                # 요약 통계
                print(f"\n• Summary:")
                if len(class_found) > 0:
                    most_common = max(class_found.items(), key=lambda x: x[1])
                    print(f"  - Most common class: {most_common[0]} ({most_common[1]:,} occurrences)")
                    
                    # 사용된 클래스 수
                    used_classes = sum(1 for v in class_found.values() if v > 0)
                    print(f"  - Classes used: {used_classes}/{len(unique_classes)}")
                    
                if undefined_values:
                    print(f"  - Undefined values: {len(undefined_values)} types, {undefined_total:,} occurrences")
            else:
                print("  - No valid data available (all values are missing)")
    
    return

# 사용 예시
if __name__ == "__main__":
    # 파일 경로 리스트
    file_paths = [
        '../../../data/kweather/version2_250813/livingroom.csv',
        '../../../data/kweather/version2_250813/room1.csv',
        '../../../data/kweather/version2_250813/room2.csv'
    ]
    
    # 각 파일 분석
    all_results = analyze_multiple_files(file_paths)

---

# 2. Plot

In [ ]:
# 파스텔톤 색상 팔레트 정의
PASTEL_COLORS = [
    '#FF91A4',  # Deep Pink
    '#6BB6E3',  # Deep Sky Blue
    '#7DD87D',  # Deep Pale Green
    '#C978C9',  # Deep Plum
    '#E6D872',  # Deep Khaki
    '#FFCF8A',  # Deep Moccasin
    '#8AC8D0',  # Deep Powder Blue
    '#E3CC9F',  # Deep Wheat
    '#C9C9E5',  # Deep Lavender
    '#FFB8B8',  # Deep Misty Rose
    '#B8E6E6',  # Deep Azure
    '#E6E680',  # Deep Light Goldenrod
    '#B798B7',  # Deep Thistle
    '#A8D8D8',  # Deep Light Cyan
    '#FFB8CD'   # Deep Lavender Blush
]


def visualize_numerical_variables(df, location_name="", imputation_method='backward'):
    """
    수치형 변수 시각화 (결측치 처리 후, 시계열 포함)
    
    Parameters:
    -----------
    df : DataFrame
        입력 데이터프레임
    location_name : str
        위치 이름
    imputation_method : str
        결측치 처리 방법 ('median', 'mean', 'forward', 'backward', 'linear', 'zero')
    """
    print("\n" + "="*80)
    print(f"📊 Numerical Variables Visualization - {location_name}")
    print("="*80)
    
    # 수치형 변수 찾기
    numerical_cols = [col for col in PREDEFINED_NUMERICAL if col in df.columns]
    
    if not numerical_cols:
        print("No numerical variables found to visualize")
        return
    
    # 결측치 처리
    df_processed = df.copy()
    
    # 시간 컬럼 확인
    if TIME_COLUMN in df_processed.columns:
        df_processed = df_processed.sort_values(TIME_COLUMN)
        
    # 결측치 위치 저장 (시각화용)
    missing_masks = {}
    for col in numerical_cols:
        missing_masks[col] = df_processed[col].isnull()
    
    print(f"\n📌 Missing Value Imputation Method: {imputation_method.upper()}")
    print("-"*50)
    
    for col in numerical_cols:
        missing_count = df_processed[col].isnull().sum()
        if missing_count > 0:
            if imputation_method == 'median':
                fill_value = df_processed[col].median()
                df_processed[col].fillna(fill_value, inplace=True)
                print(f"• {col}: {missing_count} missing values → filled with median ({fill_value:.2f})")
            
            elif imputation_method == 'mean':
                fill_value = df_processed[col].mean()
                df_processed[col].fillna(fill_value, inplace=True)
                print(f"• {col}: {missing_count} missing values → filled with mean ({fill_value:.2f})")
            
            elif imputation_method == 'forward':
                df_processed[col].fillna(method='ffill', inplace=True)
                print(f"• {col}: {missing_count} missing values → forward filled")
            
            elif imputation_method == 'backward':
                df_processed[col].fillna(method='bfill', inplace=True)
                print(f"• {col}: {missing_count} missing values → backward filled")
            
            elif imputation_method == 'linear':
                df_processed[col].interpolate(method='linear', inplace=True)
                print(f"• {col}: {missing_count} missing values → linearly interpolated")
            
            elif imputation_method == 'zero':
                df_processed[col].fillna(0, inplace=True)
                print(f"• {col}: {missing_count} missing values → filled with zero")
            
            else:
                # 기본값: median
                fill_value = df_processed[col].median()
                df_processed[col].fillna(fill_value, inplace=True)
                print(f"• {col}: {missing_count} missing values → filled with median (default)")
    
    
    # 시각화 준비
    n_cols = len(numerical_cols)
    n_rows = (n_cols + 1) // 2  # 2열로 배치
    
    # ===========================
    # Figure 1: 히스토그램과 KDE
    # ===========================
    
    fig1, axes1 = plt.subplots(n_rows, 2, figsize=(24, 6*n_rows))
    fig1.suptitle(f'Numerical Variables Distribution - {location_name}', 
                  fontsize=18, fontweight='bold', y=1.02)  # y 위치 조정
    
    if n_rows == 1:
        axes1 = axes1.reshape(1, -1)
    
    for idx, col in enumerate(numerical_cols):
        row = idx // 2
        col_idx = idx % 2
        
        # 히스토그램 with KDE
        ax = axes1[row, col_idx]
        ax.hist(df_processed[col].dropna(), bins=30, 
                color=PASTEL_COLORS[idx % len(PASTEL_COLORS)], 
                alpha=0.7, edgecolor='white', linewidth=1.2)
        
        # KDE 추가
        ax2 = ax.twinx()
        df_processed[col].plot(kind='density', ax=ax2, 
                               color=PASTEL_COLORS[(idx+5) % len(PASTEL_COLORS)], 
                               linewidth=2)
        ax2.set_ylabel('Density', fontsize=10)
        
        # 제목 설정
        ax.set_title(col, fontsize=11, fontweight='bold', pad=10)
        ax.set_xlabel('Value', fontsize=10)
        ax.set_ylabel('Frequency', fontsize=10)
        
        # 축 라벨 크기 조정
        ax.tick_params(axis='both', labelsize=9)
        ax2.tick_params(axis='y', labelsize=9)
    
    # 빈 서브플롯 숨기기
    for idx in range(n_cols, n_rows * 2):
        row = idx // 2
        col_idx = idx % 2
        axes1[row, col_idx].axis('off')
    
    # 레이아웃 조정 (더 넓은 간격)
    plt.subplots_adjust(hspace=0.2, wspace=0.3, top=1.0, bottom=0.01)
    plt.show()
    
    # ===========================
    # Figure 2: 시계열 그래프 (시간축 분할) - 결측치 표시 포함
    # ===========================
    if TIME_COLUMN in df_processed.columns:
        # 사용자가 설정 가능한 분할 row 수
        rows_per_variable = 2  # 한 변수당 시간축을 몇 개의 row로 나눌지 설정
        
        for var_idx, col in enumerate(numerical_cols):
            # 데이터 길이를 rows_per_variable로 나누기
            data_length = len(df_processed)
            chunk_size = data_length // rows_per_variable
            
            # 각 변수별로 하나의 Figure 생성
            fig2, axes2 = plt.subplots(rows_per_variable, 1, figsize=(20, 5*rows_per_variable))
            fig2.suptitle(f'Time Series Analysis: {col} - {location_name}', 
                        fontsize=16, fontweight='bold', y=1.02)
            
            # axes2를 항상 배열로 만들기
            if rows_per_variable == 1:
                axes2 = [axes2]
            
            for row_idx in range(rows_per_variable):
                ax = axes2[row_idx]
                
                # 현재 row에 해당하는 데이터 범위 계산
                if row_idx == rows_per_variable - 1:  # 마지막 row는 남은 데이터 모두 포함
                    start_idx = row_idx * chunk_size
                    end_idx = data_length
                else:
                    start_idx = row_idx * chunk_size
                    end_idx = (row_idx + 1) * chunk_size
                
                # 해당 범위의 데이터만 추출
                time_chunk = df_processed[TIME_COLUMN].iloc[start_idx:end_idx]
                data_chunk = df_processed[col].iloc[start_idx:end_idx]
                missing_chunk = missing_masks[col].iloc[start_idx:end_idx]
                
                # 시계열 플롯
                ax.plot(time_chunk, data_chunk, 
                    color=PASTEL_COLORS[var_idx % len(PASTEL_COLORS)], 
                    linewidth=1.5, alpha=0.8, label=col)
                
                # 결측치 있었던 구간 표시
                if missing_chunk.any():
                    # 결측치 구간을 음영으로 표시
                    missing_regions = []
                    in_missing = False
                    start_missing = None
                    
                    for i, is_missing in enumerate(missing_chunk):
                        if is_missing and not in_missing:
                            start_missing = i
                            in_missing = True
                        elif not is_missing and in_missing:
                            missing_regions.append((start_missing, i-1))
                            in_missing = False
                    
                    # 마지막까지 결측인 경우
                    if in_missing:
                        missing_regions.append((start_missing, len(missing_chunk)-1))
                    
                    # 음영 그리기
                    for start, end in missing_regions:
                        if start < len(time_chunk) and end < len(time_chunk):
                            ax.axvspan(time_chunk.iloc[start], time_chunk.iloc[end], 
                                     alpha=0.2, color='red', label='Missing Data' if start == missing_regions[0][0] else "")
                
                # 시간 범위 표시
                time_start = time_chunk.iloc[0] if len(time_chunk) > 0 else ""
                time_end = time_chunk.iloc[-1] if len(time_chunk) > 0 else ""
                
                # 스타일링
                ax.set_title(f'Part {row_idx+1}/{rows_per_variable}: {time_start} ~ {time_end}', 
                            fontsize=11, pad=8)
                
                # 마지막 row에만 x축 레이블 표시
                if row_idx == rows_per_variable - 1:
                    ax.set_xlabel('Time', fontsize=11)
                
                ax.set_ylabel('Value', fontsize=10)
                ax.tick_params(axis='x', rotation=45, labelsize=9)
                ax.tick_params(axis='y', labelsize=9)
                
                # 범례 추가
                ax.legend(loc='upper right', fontsize=9)
                
                # x축 날짜 포맷 개선
                ax.xaxis.set_major_locator(plt.MaxNLocator(8))
                
                # 각 구간의 통계 정보 표시
                missing_in_chunk = missing_chunk.sum()
                stats_text = f'Mean: {data_chunk.mean():.2f}, Std: {data_chunk.std():.2f}\nMissing: {missing_in_chunk} points'
                ax.text(0.02, 0.95, stats_text, transform=ax.transAxes, 
                    fontsize=8, verticalalignment='top',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
            
            plt.subplots_adjust(hspace=0.4, top=0.94, bottom=0.08)
            plt.tight_layout()
            plt.show()
    
    # ===========================
    # Figure 4: 상관관계 히트맵
    # ===========================
    fig4, ax4 = plt.subplots(1, 1, figsize=(16, 14))
    fig4.suptitle(f'Correlation Heatmap - {location_name}', 
                  fontsize=18, fontweight='bold')
    
    # 상관관계 계산
    corr_matrix = df_processed[numerical_cols].corr()
    
    # 진한 파스텔톤 커스텀 컬러맵 생성
    from matplotlib.colors import LinearSegmentedColormap
    
    # 색상 정의 (음수 -> 0 -> 양수)
    colors = ['#4A90E2',  # 진한 파란색 (강한 음의 상관)
              '#6BB6E3',  # Deep Sky Blue
              '#8AC8D0',  # Deep Powder Blue
              '#B8E6E6',  # Deep Azure
              '#E8E8E8',  # 밝은 회색 (상관관계 0 근처)
              '#FFD4D4',  # 연한 핑크
              '#FFB8B8',  # Deep Misty Rose
              '#FF91A4',  # Deep Pink
              '#FF6B9D']  # 진한 핑크 (강한 양의 상관)
    
    n_bins = 256
    deep_pastel_cmap = LinearSegmentedColormap.from_list('deep_pastel', colors, N=n_bins)
    
    # 히트맵 그리기 - 모든 셀에 값 표시 (mask 없음)
    sns.heatmap(corr_matrix, 
                annot=True,  # 모든 셀에 숫자 표시
                fmt='.2f',  # 소수점 2자리로 표시
                cmap=deep_pastel_cmap,  # 커스텀 진한 파스텔 컬러맵
                center=0,  # 0을 중심으로
                square=True,  # 정사각형 셀
                linewidths=2,  # 셀 구분선 두께
                linecolor='white',  # 흰색 구분선
                cbar_kws={
                    "shrink": 0.8, 
                    "label": "Correlation Coefficient",
                    "orientation": "vertical"
                },
                ax=ax4,
                vmin=-1, vmax=1,  # 상관계수 범위 (-1 ~ 1)
                annot_kws={
                    'size': 10,  # 글자 크기
                    'weight': 'bold',  # 굵은 글씨
                    'color': 'black'  # 검은색 글씨
                })
    
    # 제목 설정
    ax4.set_title('Pearson Correlation Coefficients\n(All values displayed)', 
                  fontsize=14, pad=20, fontweight='bold')
    
    # x축, y축 라벨 설정
    ax4.set_xticklabels(ax4.get_xticklabels(), rotation=45, ha='right', fontsize=10)
    ax4.set_yticklabels(ax4.get_yticklabels(), rotation=0, fontsize=10)
    
    # 축 라벨
    ax4.set_xlabel('Variables', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Variables', fontsize=12, fontweight='bold')
    
    plt.subplots_adjust(hspace=0.4, wspace=0.5, top=0.94, bottom=0.05)
    plt.tight_layout()
    plt.show()
    
    # ===========================
    # Figure 5: 주요 변수 통합 시계열 (시간축 분할) - 결측치 표시 포함
    # ===========================
    if TIME_COLUMN in df_processed.columns:
        # 온도와 습도 관련 변수만 선택
        temp_humidity_cols = [col for col in numerical_cols if '온도' in col or '습도' in col]
        air_quality_cols = [col for col in numerical_cols if '미세먼지' in col or '이산화탄소' in col or 'VOC' in col or '휘발성' in col]
        power_cols = [col for col in numerical_cols if '전력' in col]
        
        # 사용자가 설정 가능한 분할 row 수
        rows_per_category = 2  # 각 카테고리별 시간축을 몇 개의 row로 나눌지 설정
        
        # 데이터 길이와 청크 크기 계산
        data_length = len(df_processed)
        chunk_size = data_length // rows_per_category
        
        # 각 카테고리별로 별도의 Figure 생성
        categories = [
            ('Temperature & Humidity', temp_humidity_cols, 0),
            ('Air Quality', air_quality_cols, 5),
            ('Power Consumption', power_cols, 10)
        ]
        
        for cat_name, cat_cols, color_offset in categories:
            if cat_cols:  # 해당 카테고리에 변수가 있는 경우만
                # Figure 생성
                fig5, axes5 = plt.subplots(rows_per_category, 1, figsize=(20, 5*rows_per_category))
                fig5.suptitle(f'{cat_name} - Time Series Analysis - {location_name}', 
                              fontsize=16, fontweight='bold', y=1.02)
                
                # axes5를 항상 배열로 만들기
                if rows_per_category == 1:
                    axes5 = [axes5]
                
                for row_idx in range(rows_per_category):
                    ax = axes5[row_idx]
                    
                    # 현재 row에 해당하는 데이터 범위 계산
                    if row_idx == rows_per_category - 1:  # 마지막 row는 남은 데이터 모두 포함
                        start_idx = row_idx * chunk_size
                        end_idx = data_length
                    else:
                        start_idx = row_idx * chunk_size
                        end_idx = (row_idx + 1) * chunk_size
                    
                    # 해당 범위의 데이터만 추출
                    time_chunk = df_processed[TIME_COLUMN].iloc[start_idx:end_idx]
                    
                    # 카테고리 내 모든 변수의 결측치 위치 통합
                    any_missing = pd.Series([False] * len(time_chunk), index=time_chunk.index)
                    for col in cat_cols:
                        missing_chunk = missing_masks[col].iloc[start_idx:end_idx]
                        any_missing = any_missing | missing_chunk
                    
                    # 결측치 구간 표시 (모든 변수 통합)
                    if any_missing.any():
                        missing_regions = []
                        in_missing = False
                        start_missing = None
                        
                        for i, is_missing in enumerate(any_missing):
                            if is_missing and not in_missing:
                                start_missing = i
                                in_missing = True
                            elif not is_missing and in_missing:
                                missing_regions.append((start_missing, i-1))
                                in_missing = False
                        
                        if in_missing:
                            missing_regions.append((start_missing, len(any_missing)-1))
                        
                        # 음영 그리기
                        for start, end in missing_regions:
                            if start < len(time_chunk) and end < len(time_chunk):
                                ax.axvspan(time_chunk.iloc[start], time_chunk.iloc[end], 
                                         alpha=0.15, color='gray', zorder=0)
                    
                    # 각 변수 플롯
                    for idx, col in enumerate(cat_cols):
                        data_chunk = df_processed[col].iloc[start_idx:end_idx]
                        ax.plot(time_chunk, data_chunk, 
                               label=col,
                               color=PASTEL_COLORS[(idx + color_offset) % len(PASTEL_COLORS)], 
                               linewidth=1.5, alpha=0.8)
                    
                    # ===== Y축 스케일 설정 =====
                    from matplotlib.ticker import ScalarFormatter, LogFormatter
                    
                    if cat_name == 'Power Consumption':
                        # Power Consumption의 경우 로그 스케일 적용
                        ax.set_yscale('log')
                        
                        # 로그 스케일에서도 원본값 표시를 위한 포매터
                        y_formatter = LogFormatter(base=10, labelOnlyBase=False)
                        ax.yaxis.set_major_formatter(y_formatter)
                    else:
                        # 다른 카테고리는 원본값 표시 설정
                        # 과학적 표기법 비활성화
                        ax.ticklabel_format(axis='y', style='plain', useOffset=False)
                        
                        # Y축 포매터 설정 (원본 값 그대로 표시)
                        y_formatter = ScalarFormatter(useOffset=False)
                        y_formatter.set_scientific(False)
                        ax.yaxis.set_major_formatter(y_formatter)
                    # ================================
                    
                    # 시간 범위 표시
                    time_start = time_chunk.iloc[0] if len(time_chunk) > 0 else ""
                    time_end = time_chunk.iloc[-1] if len(time_chunk) > 0 else ""
                    
                    # 스타일링
                    missing_count = any_missing.sum()
                    title_text = f'Part {row_idx+1}/{rows_per_category}: {time_start} ~ {time_end}'
                    if missing_count > 0:
                        title_text += f' (Missing: {missing_count} points)'
                    ax.set_title(title_text, fontsize=11, pad=8)
                    
                    # 마지막 row에만 x축 레이블 표시
                    if row_idx == rows_per_category - 1:
                        ax.set_xlabel('Time', fontsize=11)
                    
                    # Y축 레이블 설정
                    if cat_name == 'Power Consumption':
                        ax.set_ylabel('Value (log scale)', fontsize=11)
                    else:
                        ax.set_ylabel('Value', fontsize=11)
                    
                    # 범례 수정 (결측치 표시 추가)
                    handles, labels = ax.get_legend_handles_labels()
                    if any_missing.any():
                        gray_patch = plt.Rectangle((0, 0), 1, 1, fc='gray', alpha=0.15, label='Missing Data Region')
                        handles.append(gray_patch)
                        labels.append('Missing Data Region')
                    ax.legend(handles=handles, labels=labels, loc='upper right', ncol=min(3, len(labels)), fontsize=9)
                    
                    ax.tick_params(axis='x', rotation=45, labelsize=9)
                    ax.tick_params(axis='y', labelsize=9)
                    
                    # x축 날짜 포맷 개선
                    ax.xaxis.set_major_locator(plt.MaxNLocator(10))
                
                plt.subplots_adjust(hspace=0.4, top=0.94, bottom=0.08)
                plt.tight_layout()
                plt.show()
    
    return df_processed

def visualize_categorical_variables(df, location_name=""):
    """
    범주형 변수의 클래스 분포 시각화
    """
    print("\n" + "="*80)
    print(f"📊 Categorical Variables Visualization - {location_name}")
    print("="*80)
    
    # 범주형 변수 찾기
    categorical_cols = [col for col in PREDEFINED_CATEGORICAL if col in df.columns]
    
    if not categorical_cols:
        print("No categorical variables found to visualize")
        return
    
    # Figure 1: 각 변수별 클래스 분포
    n_cols = len(categorical_cols)
    n_rows = (n_cols + 1) // 2  # 2열로 배치
    
    fig1, axes1 = plt.subplots(n_rows, 2, figsize=(16, 5*n_rows))
    fig1.suptitle(f'Categorical Variables Class Distribution - {location_name}', 
                  fontsize=16, fontweight='bold')
    
    if n_rows == 1:
        axes1 = axes1.reshape(1, -1)
    
    for idx, col in enumerate(categorical_cols):
        row = idx // 2
        col_idx = idx % 2
        ax = axes1[row, col_idx]
        
        # 값 빈도 계산
        value_counts = df[col].value_counts()
        
        # 파이 차트 그리기
        colors = PASTEL_COLORS[:len(value_counts)]
        wedges, texts, autotexts = ax.pie(value_counts.values, 
                                           labels=value_counts.index,
                                           colors=colors,
                                           autopct='%1.1f%%',
                                           startangle=90)
        
        # 텍스트 스타일링
        for text in texts:
            text.set_fontsize(10)
        for autotext in autotexts:
            autotext.set_color('black')
            autotext.set_fontsize(9)
            autotext.set_weight('bold')
        
        ax.set_title(f'{col}\n(Total: {value_counts.sum()} valid values)', 
                    fontsize=12, fontweight='bold')
    
    # 빈 서브플롯 숨기기
    if n_cols % 2 == 1:
        axes1[n_rows-1, 1].axis('off')
    
    plt.tight_layout()
    
    # Figure 2: 전체 범주형 변수 요약
    fig2, axes2 = plt.subplots(2, 1, figsize=(14, 12))
    fig2.suptitle(f'Categorical Variables Summary - {location_name}', 
                  fontsize=16, fontweight='bold')
    
    # 상단: 각 변수의 고유값 개수
    ax = axes2[0]
    unique_counts = [df[col].nunique() for col in categorical_cols]
    bars = ax.bar(range(len(categorical_cols)), unique_counts, 
                  color=PASTEL_COLORS[:len(categorical_cols)])
    
    # 막대 위에 값 표시
    for i, (bar, count) in enumerate(zip(bars, unique_counts)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{count}', ha='center', va='bottom', fontweight='bold')
    
    ax.set_xticks(range(len(categorical_cols)))
    ax.set_xticklabels(categorical_cols, rotation=45, ha='right')
    ax.set_ylabel('Number of Unique Classes')
    ax.set_title('Number of Unique Classes per Variable', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 하단: 결측치 비율
    ax = axes2[1]
    missing_ratios = [(df[col].isnull().sum() / len(df)) * 100 for col in categorical_cols]
    bars = ax.bar(range(len(categorical_cols)), missing_ratios,
                  color=PASTEL_COLORS[5:5+len(categorical_cols)])
    
    # 막대 위에 값 표시
    for bar, ratio in zip(bars, missing_ratios):
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{ratio:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    ax.set_xticks(range(len(categorical_cols)))
    ax.set_xticklabels(categorical_cols, rotation=45, ha='right')
    ax.set_ylabel('Missing Ratio (%)')
    ax.set_title('Missing Value Ratio per Variable', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    # Figure 3: 정의된 클래스 vs 실제 데이터 비교
    fig3, axes3 = plt.subplots(2, 3, figsize=(18, 10))
    fig3.suptitle(f'Defined Classes vs Actual Data - {location_name}', 
                  fontsize=16, fontweight='bold')
    
    for idx, col in enumerate(categorical_cols[:6]):  # 최대 6개만
        row = idx // 3
        col_idx = idx % 3
        ax = axes3[row, col_idx]
        
        if col in CATEGORICAL_CLASSES:
            defined_classes = CATEGORICAL_CLASSES[col]
            unique_classes = list(set(defined_classes.values()))
            
            # 실제 데이터에서 각 클래스 카운트
            class_counts = {}
            for class_name in unique_classes:
                # 해당 클래스에 매칭되는 키들 찾기
                matching_keys = [k for k, v in defined_classes.items() if v == class_name]
                count = 0
                for key in matching_keys:
                    count += (df[col].astype(str) == str(key)).sum()
                class_counts[class_name] = count
            
            # 막대 그래프
            bars = ax.bar(range(len(class_counts)), list(class_counts.values()),
                         color=PASTEL_COLORS[:len(class_counts)])
            
            # 값 표시
            for bar, (name, count) in zip(bars, class_counts.items()):
                height = bar.get_height()
                if height > 0:
                    ax.text(bar.get_x() + bar.get_width()/2., height,
                           f'{count}', ha='center', va='bottom', fontsize=9)
            
            ax.set_xticks(range(len(class_counts)))
            ax.set_xticklabels(list(class_counts.keys()), rotation=45, ha='right', fontsize=8)
            ax.set_ylabel('Count')
            ax.set_title(f'{col}', fontsize=10, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')
    
    # 빈 서브플롯 숨기기
    for idx in range(len(categorical_cols), 6):
        row = idx // 3
        col_idx = idx % 3
        axes3[row, col_idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    """
    범주형 변수의 클래스 분포 시각화
    """
    print("\n" + "="*80)
    print(f"📊 Categorical Variables Visualization - {location_name}")
    print("="*80)
    
    # 범주형 변수 찾기
    categorical_cols = [col for col in PREDEFINED_CATEGORICAL if col in df.columns]
    
    if not categorical_cols:
        print("No categorical variables found to visualize")
        return
    
    # Figure 1: 각 변수별 클래스 분포
    n_cols = len(categorical_cols)
    n_rows = (n_cols + 1) // 2  # 2열로 배치
    
    fig1, axes1 = plt.subplots(n_rows, 2, figsize=(16, 5*n_rows))
    fig1.suptitle(f'Categorical Variables Class Distribution - {location_name}', 
                  fontsize=16, fontweight='bold')
    
    if n_rows == 1:
        axes1 = axes1.reshape(1, -1)
    
    for idx, col in enumerate(categorical_cols):
        row = idx // 2
        col_idx = idx % 2
        ax = axes1[row, col_idx]
        
        # 값 빈도 계산
        value_counts = df[col].value_counts()
        
        # 파이 차트 그리기
        colors = PASTEL_COLORS[:len(value_counts)]
        wedges, texts, autotexts = ax.pie(value_counts.values, 
                                           labels=value_counts.index,
                                           colors=colors,
                                           autopct='%1.1f%%',
                                           startangle=90)
        
        # 텍스트 스타일링
        for text in texts:
            text.set_fontsize(10)
        for autotext in autotexts:
            autotext.set_color('black')
            autotext.set_fontsize(9)
            autotext.set_weight('bold')
        
        ax.set_title(f'{col}\n(Total: {value_counts.sum()} valid values)', 
                    fontsize=12, fontweight='bold')
    
    # 빈 서브플롯 숨기기
    if n_cols % 2 == 1:
        axes1[n_rows-1, 1].axis('off')
    
    plt.tight_layout()
    
    # Figure 2: 전체 범주형 변수 요약
    fig2, axes2 = plt.subplots(2, 1, figsize=(14, 12))
    fig2.suptitle(f'Categorical Variables Summary - {location_name}', 
                  fontsize=16, fontweight='bold')
    
    # 상단: 각 변수의 고유값 개수
    ax = axes2[0]
    unique_counts = [df[col].nunique() for col in categorical_cols]
    bars = ax.bar(range(len(categorical_cols)), unique_counts, 
                  color=PASTEL_COLORS[:len(categorical_cols)])
    
    # 막대 위에 값 표시
    for i, (bar, count) in enumerate(zip(bars, unique_counts)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{count}', ha='center', va='bottom', fontweight='bold')
    
    ax.set_xticks(range(len(categorical_cols)))
    ax.set_xticklabels(categorical_cols, rotation=45, ha='right')
    ax.set_ylabel('Number of Unique Classes')
    ax.set_title('Number of Unique Classes per Variable', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 하단: 결측치 비율
    ax = axes2[1]
    missing_ratios = [(df[col].isnull().sum() / len(df)) * 100 for col in categorical_cols]
    bars = ax.bar(range(len(categorical_cols)), missing_ratios,
                  color=PASTEL_COLORS[5:5+len(categorical_cols)])
    
    # 막대 위에 값 표시
    for bar, ratio in zip(bars, missing_ratios):
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{ratio:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    ax.set_xticks(range(len(categorical_cols)))
    ax.set_xticklabels(categorical_cols, rotation=45, ha='right')
    ax.set_ylabel('Missing Ratio (%)')
    ax.set_title('Missing Value Ratio per Variable', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    # Figure 3: 정의된 클래스 vs 실제 데이터 비교
    fig3, axes3 = plt.subplots(2, 3, figsize=(18, 10))
    fig3.suptitle(f'Defined Classes vs Actual Data - {location_name}', 
                  fontsize=16, fontweight='bold')
    
    for idx, col in enumerate(categorical_cols[:6]):  # 최대 6개만
        row = idx // 3
        col_idx = idx % 3
        ax = axes3[row, col_idx]
        
        if col in CATEGORICAL_CLASSES:
            defined_classes = CATEGORICAL_CLASSES[col]
            unique_classes = list(set(defined_classes.values()))
            
            # 실제 데이터에서 각 클래스 카운트
            class_counts = {}
            for class_name in unique_classes:
                # 해당 클래스에 매칭되는 키들 찾기
                matching_keys = [k for k, v in defined_classes.items() if v == class_name]
                count = 0
                for key in matching_keys:
                    count += (df[col].astype(str) == str(key)).sum()
                class_counts[class_name] = count
            
            # 막대 그래프
            bars = ax.bar(range(len(class_counts)), list(class_counts.values()),
                         color=PASTEL_COLORS[:len(class_counts)])
            
            # 값 표시
            for bar, (name, count) in zip(bars, class_counts.items()):
                height = bar.get_height()
                if height > 0:
                    ax.text(bar.get_x() + bar.get_width()/2., height,
                           f'{count}', ha='center', va='bottom', fontsize=9)
            
            ax.set_xticks(range(len(class_counts)))
            ax.set_xticklabels(list(class_counts.keys()), rotation=45, ha='right', fontsize=8)
            ax.set_ylabel('Count')
            ax.set_title(f'{col}', fontsize=10, fontweight='bold')
    
    # 빈 서브플롯 숨기기
    for idx in range(len(categorical_cols), 6):
        row = idx // 3
        col_idx = idx % 3
        axes3[row, col_idx].axis('off')
    
    plt.tight_layout()
    plt.show()


In [ ]:
for location, result in all_results.items():
    # 수치형 변수 시각화
    df_processed = visualize_numerical_variables(result['dataframe'], location)
    
    # 범주형 변수 시각화
    visualize_categorical_variables(result['dataframe'], location)